In [110]:
import numpy as np
from numpy.polynomial.polynomial import polyfit, polyval
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd

np.random.seed(42)

In [111]:
S0 = 100
K = 100
r = 0.02
u = 1.2
d = 0.8
T = 3

n_steps = 3
n_simulations = 100000

**American Call Option Pricing**

The value can be calculated by means of a three, from the final payoff, knowing the price movement one can compare them to the payoff at each step, given that the American Opt give the right to exercise in each moment during the period.

call = max(S_t-K, discouted expectation of next continuation value that in the end is the final payoff, regression method)

In [112]:
# function that is used in the binomial model to price the previous node give the next 2 price (t+1) and the actual price of the stock in t.

# we need to use the risk neutral measure q, found by solving a lineare system given by 2 equations
# (1+r) = q_u * u + q_d * d 
# 1 = q_u + q_d

def succ_el(node,u,d):
    return node*u, node*d

def three_generation(n_step, u=1.2, d=0.8):
    S = 100
    three = [[S]]
    for i in range(1,n_step):
        new_node = []
        for el in three[i-1]:
            new_elements_up, new_elements_down = succ_el(el,u,d)
            if len(new_node) == 0:
                new_node += [new_elements_up,new_elements_down]
            else:    
                new_node += [new_elements_down]
        three.append(new_node)

    return three
        

In [113]:
# we need to go from the price three to the payoff three

three = three_generation(n_steps+1)

payoff_three = [row[:] for row in three]
payoff_three[-1] = [max(S-K,0) for S in three[-1]]

In [114]:
q_u = ((1+r) - d)/(u-d)
q_d = 1-q_u


for i in range(len(three)-1,0,-1):
    l = []
    for ind,el in enumerate(three[i-1]):
        continuation = (1/(1+r)) * (q_u * payoff_three[i][ind] + q_d * payoff_three[i][ind+1])
        early_exercise = max(el-K, 0)
        l.append(max(early_exercise,continuation))
    payoff_three[i-1] = l

print("American Call Price in t=0: ", round(payoff_three[0][0],3))

American Call Price in t=0:  17.263


Let's try to verify this value versus the **Longstaff_Schwartz** monte-carlo pricing.

In [115]:
def simulate_gbm_paths(S0, T, r, sigma, n_steps, n_simulations):
    dt = T / n_steps

    z = np.random.normal(0, 1, size=(n_simulations, n_steps))
    increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z

    log_paths = np.cumsum(increments, axis=1)
    log_paths = np.hstack([np.zeros((n_simulations, 1)), log_paths])

    paths = S0 * np.exp(log_paths)
    return paths

def american_call_payoff(S,K):
    return np.maximum(S-K,0)

In [124]:
def longstaff_schwartz_put(S0, K, r, sigma, T, step, n_path):
    dt = T / step
    discount = np.exp(-r*dt)

    paths = simulate_gbm_paths(S0, T, r, sigma, step, n_path)

    cashflows = american_call_payoff(paths[:,-1], K)

    for t in range(step-1, 0, -1):
        St = paths[:,t]
        exercise_value = american_call_payoff(St, K)

        # use only ITM options because they are the only one where the exercise decision is to be made or not
        # OTM path could introduce bias towards 0
        in_the_money = exercise_value > 0

        X = St[in_the_money]
        Y = cashflows[in_the_money] * discount
    
        if len(X) > 0:
            # trying to fit X to Y because X is the today available information and we want ot fit it to the future continuation value from simulation
            # looking for the future mean given the present value X ( E[ Y | X ] )
            coeffs = polyfit(X,Y,deg=2)
            continuation_value = polyval(St, coeffs)

            exercise_now = exercise_value > continuation_value

            # where continuation is > substitute the immediate payoff
            cashflows[exercise_now] = exercise_value[exercise_now]
            cashflows[~exercise_now] = cashflows[~exercise_now] * discount

    price = np.mean(cashflows) * discount

    return price


In [143]:
price = longstaff_schwartz_put(
    S0=S0,
    K=K,
    r=r,
    sigma=sigma,
    T=T,
    step=n_steps,
    n_path=n_simulations
)

print(f"American call option price using LS at t=0: {round(price,3)}")

American call option price using LS at t=0: 15.602
